<a href="https://colab.research.google.com/github/hinojosaM/machine_learning_homework5/blob/main/machine_learning_homework5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import hashlib
import json
import os
import pickle
import re
import subprocess
import time
import urllib.request

In [14]:
REPO_URL = "https://github.com/DataTalksClub/machine-learning-zoomcamp.git"
REPO_DIR = "machine-learning-zoomcamp"
HW_REL_PATH = "cohorts/2026/homework/05-deployment"

if not os.path.exists(REPO_DIR):
    print("Clonando el repositorio de ML Zoomcamp...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)

hw_dir = os.path.abspath(os.path.join(REPO_DIR, HW_REL_PATH))
os.chdir(hw_dir)
print(f"Directorio de trabajo actual: {os.getcwd()}\n")

print("Instalando uv...")
subprocess.run(["pip", "install", "-q", "uv"], check=True)

# Verificar el checksum SHA-256 de pipeline.bin
EXPECTED_SHA256 = "1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1"
with open("pipeline.bin", "rb") as f:
    actual_sha256 = hashlib.sha256(f.read()).hexdigest()

print(f"SHA256 pipeline.bin: {actual_sha256}")
assert actual_sha256 == EXPECTED_SHA256, "ERROR: El checksum de pipeline.bin no coincide."
print("Checksum verificado correctamente.\n")

# Sincronizar el entorno bloqueado (Python 3.11 + uv.lock) y ejecutar smoke_test.py
print("Sincronizando dependencias con `uv sync --locked`...")
subprocess.run(["uv", "sync", "--locked"], check=True)

print("Ejecutando smoke_test.py...")
smoke_res = subprocess.run(["uv", "run", "--locked", "python", "smoke_test.py"], capture_output=True, text=True, check=True)
print(smoke_res.stdout)


Clonando el repositorio de ML Zoomcamp...
Directorio de trabajo actual: /content/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment

Instalando uv...
SHA256 pipeline.bin: 1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1
Checksum verificado correctamente.

Sincronizando dependencias con `uv sync --locked`...
Ejecutando smoke_test.py...
{'conversion_probability': 0.769799, 'conversion': True}



In [15]:
uv_ver = subprocess.run(["uv", "--version"], capture_output=True, text=True, check=True).stdout.strip()

print(f"Q 1 (Setup check - uv version): {uv_ver}")


Q 1 (Setup check - uv version): uv 0.13.5 (x86_64-unknown-linux-gnu)


In [16]:
with open("pyproject.toml", "r", encoding="utf-8") as f:
    pyproject_text = f.read()

with open("uv.lock", "r", encoding="utf-8") as f:
    uv_lock_text = f.read()

sklearn_lock_match = re.search(
    r'name\s*=\s*"scikit-learn"\s*\nversion\s*=\s*"([^"]+)"',
    uv_lock_text
)
sklearn_version = sklearn_lock_match.group(1) if sklearn_lock_match else "No encontrado"

print(f"\nQ 2 (Scikit-Learn version en uv.lock): {sklearn_version}")
print("Contenido relevante de pyproject.toml:")
for line in pyproject_text.splitlines():
    if "scikit-learn" in line.lower() or "python" in line.lower():
        print(f"  {line}")



Q 2 (Scikit-Learn version en uv.lock): 1.7.2
Contenido relevante de pyproject.toml:
  requires-python = ">=3.11,<3.12"
      "scikit-learn==1.7.2",


In [17]:
q3_code = """
import pickle

lead_q3 = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41
}

with open("pipeline.bin", "rb") as f_in:
    pipeline = pickle.load(f_in)

prob = float(pipeline.predict_proba([lead_q3])[0, 1])
print(f"Probabilidad exacta: {prob:.6f}")
print(f"Respuesta redondeada (3 decimales): {prob:.3f}")
"""

q3_res = subprocess.run(
    ["uv", "run", "--locked", "python", "-c", q3_code],
    capture_output=True,
    text=True,
    check=True
)
print("\nQ 3 (Inferencia directa con pipeline.bin):")
print(q3_res.stdout.strip())


Q 3 (Inferencia directa con pipeline.bin):
Probabilidad exacta: 0.532994
Respuesta redondeada (3 decimales): 0.533


In [18]:
import json
import subprocess
import time
import urllib.request

print("Iniciando servidor FastAPI en segundo plano (puerto 9696)...")
server_proc = subprocess.Popen(
    ["uv", "run", "--locked", "uvicorn", "predict:app", "--host", "0.0.0.0", "--port", "9696"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

health_data = None
for _ in range(30):
    try:
        with urllib.request.urlopen("http://localhost:9696/health", timeout=1) as resp:
            if resp.status == 200:
                health_data = json.loads(resp.read().decode("utf-8"))
                break
    except Exception:
        time.sleep(0.5)

lead_q4 = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74
}

req = urllib.request.Request(
    "http://localhost:9696/predict",
    data=json.dumps(lead_q4).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST"
)
with urllib.request.urlopen(req, timeout=5) as resp:
    q4_response = json.loads(resp.read().decode("utf-8"))

prob_q4 = float(q4_response.get("conversion_probability", 0.0))
print("=" * 60)
print("Q 4 (POST /predict):")
print(f"Respuesta JSON completa: {json.dumps(q4_response)}")
print(f"conversion_probability (3 decimales): {prob_q4:.3f}")



Iniciando servidor FastAPI en segundo plano (puerto 9696)...
Q 4 (POST /predict):
Respuesta JSON completa: {"conversion_probability": 0.769799, "conversion": true}
conversion_probability (3 decimales): 0.770


In [19]:
with open("Dockerfile", "r", encoding="utf-8") as f:
    dockerfile_text = f.read()

from_lines = [line.strip() for line in dockerfile_text.splitlines() if line.strip().upper().startswith("FROM")]
print("=" * 60)
print("Q 5 (Imagen base declarada en Dockerfile):")
for line in from_lines:
    print(f"  {line}")
print("Respuesta: python:3.11.15-slim-bookworm")


Q 5 (Imagen base declarada en Dockerfile):
  FROM python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3
Respuesta: python:3.11.15-slim-bookworm


In [20]:
import json
import os
import subprocess

with open("model_metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("=" * 60)
print("Q 6 (Prueba del servicio y verificación de /health):")
print(f"Respuesta GET /health: {json.dumps(health_data)}")
print(f"model_metadata.json:   {json.dumps(metadata)}")

if os.path.exists("q6_test.py"):
    q6_res = subprocess.run(
        ["uv", "run", "--locked", "python", "q6_test.py"],
        capture_output=True,
        text=True,
        check=True
    )
    print("Salida de q6_test.py:")
    print(q6_res.stdout.strip())

print(f"conversion_probability (3 decimales, igual que Q4): {prob_q4:.3f}")
print("=" * 60)

server_proc.terminate()
server_proc.wait(timeout=5)
print("Servidor FastAPI detenido correctamente.")

Q 6 (Prueba del servicio y verificación de /health):
Respuesta GET /health: {"status": "ok", "model_sha256": "1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1", "feature_defaults_sha256": "1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd"}
model_metadata.json:   {"dataset": "course_lead_scoring_2026.csv", "dataset_sha256": "2d9da196bdefd2a45aa0a17ca21c19f06a5d9d56af628be1f04bbb3531cc12e9", "rows": 5000, "features": ["lead_source", "industry", "employment_status", "location", "number_of_courses_viewed", "annual_income", "interaction_count", "lead_score"], "target": "converted", "model": "DictVectorizer + LogisticRegression(solver=liblinear, C=1.0)", "random_state": 42, "scikit_learn": "1.7.2", "numeric_imputation": "training-set median", "numeric_defaults": {"number_of_courses_viewed": 2.0, "annual_income": 55051.0, "interaction_count": 4.0, "lead_score": 0.49}, "model_sha256": "1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1", "feature_d